# Mobility

This notebook prepares the municipality-level commuting flows dataset used by HERMES.

## Pipeline

1. Download dataset
2. Load dataset
3. Explore raw dataset
4. Validate raw dataset
5. Prepare dataset
6. Explore prepared dataset
7. Validate prepared dataset
8. Save dataset

## Output

- `mobility.parquet`

In [ ]:
# ============================================================================
# Imports
# ============================================================================

from hermes.config import PREPARED_DIR
from hermes.io import save_dataframe
from hermes.loaders import load_mobility_raw
from hermes.preprocessing import prepare_mobility
from hermes.sources import download_mobility
from hermes.validation import (
    explore_dataframe,
    validate_dataframe,
)

In [ ]:
# ============================================================================
# Download dataset
# ============================================================================

download_mobility()

In [ ]:
# ============================================================================
# Load dataset
# ============================================================================

mobility_raw = load_mobility_raw()

In [ ]:
# ============================================================================
# Explore raw dataset
# ============================================================================

explore_dataframe(
    mobility_raw,
    title="Raw Mobility Dataset",
)

In [ ]:
# ============================================================================
# Validate raw dataset
# ============================================================================

validate_dataframe(
    mobility_raw,
    expect_missing=True,
)

In [ ]:
(
    mobility_raw
    .isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)

In [ ]:
print(mobility_raw.columns.tolist())

In [ ]:
mobility_raw[
    mobility_raw["CODGEO"].isna()
]

<span style="color:red">**TODO**  
Investigate missing origin municipalities.  
The source dataset contains 12 rows with missing CODGEO/LIBGEO.</span>  

In [ ]:
mobility_raw.head()

In [ ]:
mobility_raw.columns.tolist()

In [ ]:
# ============================================================================
# Prepare dataset
# ============================================================================

mobility = prepare_mobility(
    mobility_raw,
)

In [ ]:
mobility.columns.tolist()

In [ ]:
# ============================================================================
# Explore prepared dataset
# ============================================================================

explore_dataframe(
    mobility,
    title="Prepared Mobility Dataset",
)

In [ ]:
mobility.loc[
    mobility["origin_insee_code"].isna()
    | mobility["destination_insee_code"].isna()
]

In [ ]:
mobility_raw.loc[
    mobility_raw.iloc[:, 0].isna()
]

In [ ]:
mobility_raw.iloc[1003625:1003647]

In [ ]:
mobility_raw.tail(12)

In [ ]:
mobility_raw.info()

In [ ]:
# ============================================================================
# Validate prepared dataset
# ============================================================================

validate_dataframe(
    mobility,
    key=["origin_insee_code", "destination_insee_code",],
    expect_missing=True,
)

<span style="color:red">
<b>Known issue: 12 mobility records with missing origin municipality</b>

During validation of the official INSEE 2023 mobility dataset, 12 records were found with missing origin municipality information (`CODGEO` and `LIBGEO`).

Characteristics:

- Present in the official CSV file.
- Located at the end of the dataset.
- Destination municipality is available.
- Commuter flow is non-zero.
- No explanation was found in the official methodological [documentation (June 2022)](https://www.insee.fr/fr/statistiques/8998300?#documentation).

These records are preserved in the prepared dataset to remain faithful to the official source. They cannot currently be represented in the territorial graph because an edge requires both an origin and a destination node.

This issue will be investigated in a future version of HERMES.
</span>

In [ ]:
# ============================================================================
# Save dataset
# ============================================================================

save_dataframe(
    mobility,
    PREPARED_DIR / "mobility.parquet",
)